# Qwen 14B trên Colab free (Ollama) cho MungNgoc

Chạy lần lượt từng ô, **bật GPU trước**: *Runtime → Change runtime type → T4 GPU*.
Ô cuối in ra một địa chỉ `https://....trycloudflare.com`. Dán địa chỉ đó vào `OLLAMA_HOST` trong `.env` trên máy bạn
(hoặc truyền `--host`), rồi chạy `python3 -m qwen_translate sample/srt`.

**Lưu ý:** địa chỉ này công khai và không có mật khẩu (chỉ ai biết đúng chuỗi ngẫu nhiên mới dùng được). Đừng đăng ở đâu cả, tắt runtime khi xong.

In [ ]:
# 1. Kiểm tra GPU (cần thấy Tesla T4 ~15GB)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cài Ollama
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Chạy server Ollama ở nền
import os, subprocess, time, requests
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="24h")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2); print("Ollama đã chạy."); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama không khởi động được, xem /content/ollama.log")

In [ ]:
# 4. Tải model (~9GB, vài phút). Đổi tên nếu muốn model khác.
MODEL = "qwen2.5:14b-instruct"
!ollama pull $MODEL

In [ ]:
# 5. Thử nhanh model (phải ra được một câu tiếng Việt)
r = requests.post("http://127.0.0.1:11434/api/generate", json={"model": MODEL, "prompt": "Dịch sang tiếng Việt: 师傅，你的新衣服真好看。", "stream": False}, timeout=300)
print(r.json()["response"])

In [ ]:
# 6. Mở tunnel Cloudflare (không cần tài khoản) và in địa chỉ
import re, threading
!curl -fsSL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "localhost:11434"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break
print("\nOLLAMA_HOST =", public_url)
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # đọc hết log để tunnel không bị nghẽn

In [ ]:
# 7. Giữ phiên sống trong lúc dịch. Dừng ô này (nút Stop) khi dịch xong rồi tắt runtime.
import time
print("Đang chờ... địa chỉ:", public_url)
while True:
    time.sleep(60)
    try:
        requests.get(public_url + "/api/tags", timeout=10)
    except Exception as e:
        print("Tunnel không phản hồi:", e)